# Aula 25: Árvores de Comportamento (Behavior Trees) — Controle Reativo e Tomada de Decisão do Visão-AGV

## 1. Visão Geral da Aula

Nesta aula, implementaremos a estrutura de uma **Árvore de Comportamento (Behavior Tree - BT)** para orquestrar a tomada de decisão reativa e o controle de segurança do sistema **Visão-AGV** em tempo real.

### Contexto no Projeto Visão-AGV
Diferente das máquinas de estados finitos (FSM), as Árvores de Comportamento oferecem modularidade e reatividade para reagir instantaneamente a eventos críticos no ambiente da fábrica (ex: travamento de emergência por obstáculo repentino ou bateria fraca) sem perder o contexto da tarefa principal (ex: transporte de palete).

**Tipos de Nós Implementados:**
1. **Nós de Controle (Compostos):**
   - **Seletor / Fallback (`?`):** Executa os filhos até que um retorne `SUCCESS`. Ideal para priorização de segurança e falhas.
   - **Sequência (`->`):** Executa os filhos em ordem até que todos retornem `SUCCESS`. Usado para rotinas compostas de navegação e carga.
2. **Nós Folha:**
   - **Condições (Checagens):** Retornam `SUCCESS` ou `FAILURE` (ex: `BateriaCritica?`, `ObstaculoDetectado?`).
   - **Ações (Executores):** Realizam tarefas e retornam `RUNNING`, `SUCCESS` ou `FAILURE` (ex: `NavegarParaDoca`, `ParadaEmergencia`).


In [ ]:
# Framework Base para Árvores de Comportamento (Behavior Tree)
from enum import Enum

class StatusNo(Enum):
    SUCCESS = 1
    FAILURE = 2
    RUNNING = 3

class NoBT:
    """Classe base abstrata para todos os nós da Behavior Tree."""
    def tick(self, contexto_agv):
        raise NotImplementedError("Cada nó deve implementar seu próprio método tick().")

class NoSeletor(NoBT):
    """Nó Seletor (Fallback): Retorna SUCCESS assim que um filho retornar SUCCESS."""
    def __init__(self, nome, filhos):
        self.nome = nome
        self.filhos = filhos

    def tick(self, contexto_agv):
        for filho in self.filhos:
            status = filho.tick(contexto_agv)
            if status != StatusNo.FAILURE:
                return status
        return StatusNo.FAILURE

class NoSequencia(NoBT):
    """Nó Sequência: Retorna FAILURE assim que um filho retornar FAILURE."""
    def __init__(self, nome, filhos):
        self.nome = nome
        self.filhos = filhos

    def tick(self, contexto_agv):
        for filho in self.filhos:
            status = filho.tick(contexto_agv)
            if status != StatusNo.SUCCESS:
                return status
        return StatusNo.SUCCESS

print("Classes base e compostas da Behavior Tree carregadas!")

## 2. Implementação das Condições e Ações Específicas do Visão-AGV

Criamos os nós folha que interagem diretamente com os dados de telemetria (nível de bateria, sensores LiDAR de proximidade e ordens de transporte).

In [ ]:
# Implementação dos Nós Folha (Condições e Ações)

class CondicaoObstaculo(NoBT):
    def tick(self, contexto):
        if contexto['obstaculo_distancia_m'] < 1.0:
            print("  [Checagem] ⚠️ OBSTÁCULO PRÓXIMO DETECTADO!")
            return StatusNo.SUCCESS
        return StatusNo.FAILURE

class AcaoParadaEmergencia(NoBT):
    def tick(self, contexto):
        print("  [AÇÃO CRÍTICA] 🛑 E-STOP ATIVADO! AGV Parado com segurança.")
        contexto['estado_agv'] = 'PARADA_EMERGENCIA'
        return StatusNo.SUCCESS

class CondicaoBateriaBaixa(NoBT):
    def tick(self, contexto):
        if contexto['bateria_percentual'] < 15.0:
            print("  [Checagem] 🪫 Bateria crítica (<15%).")
            return StatusNo.SUCCESS
        return StatusNo.FAILURE

class AcaoRecarregar(NoBT):
    def tick(self, contexto):
        print("  [Ação] 🔌 Navegando até a Estação de Recarga ST-01...")
        contexto['estado_agv'] = 'RECARGANDO'
        return StatusNo.SUCCESS

class AcaoNavegarETransportar(NoBT):
    def tick(self, contexto):
        if contexto.get('tarefa_pendente'):
            print(f"  [Ação] 🚚 Executando transporte: {contexto['tarefa_pendente']}")
            contexto['estado_agv'] = 'EM_TRANSITO'
            return StatusNo.SUCCESS
        print("  [Ação] 💤 Nenhum transporte pendente. AGV em Standby.")
        return StatusNo.SUCCESS

print("Nós de condição e ação do Visão-AGV prontos!")

## 3. Construção e Simulação da Árvore de Decisão do AGV

A árvore é montada com uma hierarquia de prioridades:
1. **Prioridade 1 (Segurança Extrema):** Se houver obstáculo próximo $\rightarrow$ Ativa E-STOP.
2. **Prioridade 2 (Manutenção):** Se bateria $< 15\%$ $\rightarrow$ Vai para Recarga.
3. **Prioridade 3 (Operação Normal):** Caso contrário $\rightarrow$ Executa a tarefa de navegação e transporte.


In [ ]:
# Montagem da Árvore de Comportamento Principal (Root)
sub_emergencia = NoSequencia("Rotina Emergência", [
    CondicaoObstaculo(),
    AcaoParadaEmergencia()
])

sub_recarga = NoSequencia("Rotina Recarga", [
    CondicaoBateriaBaixa(),
    AcaoRecarregar()
])

arvore_raiz = NoSeletor("Raiz Controle AGV", [
    sub_emergencia,
    sub_recarga,
    AcaoNavegarETransportar()
])

# --- SIMULAÇÃO DE MÚLTIPLOS CICLOS DE EXECUÇÃO (TICKS) ---
cenarios_telemetria = [
    {'nome': 'Cenário 1: Operação Normal', 'obstaculo_distancia_m': 3.5, 'bateria_percentual': 85.0, 'tarefa_pendente': 'ORD-101 -> DOC-101'},
    {'nome': 'Cenário 2: Presença de Obstáculo', 'obstaculo_distancia_m': 0.4, 'bateria_percentual': 80.0, 'tarefa_pendente': 'ORD-101 -> DOC-101'},
    {'nome': 'Cenário 3: Bateria Crítica sem Obstáculo', 'obstaculo_distancia_m': 4.0, 'bateria_percentual': 10.0, 'tarefa_pendente': 'ORD-102 -> ALM-201'}
]

for sim in cenarios_telemetria:
    print(f"\n=== {sim['nome']} ===")
    status_final = arvore_raiz.tick(sim)
    print(f"Status Final da Árvore: {status_final}")
    print(f"Estado resultante do AGV: {sim.get('estado_agv', 'INALTERADO')}")

## 4. Conclusão e Análise Comportamental

1. **Reatividade em Tempo Real:** A estrutura hierárquica garante que a rotina de segurança sobreponha imediatamente qualquer ordem de movimentação operacional.
2. **Extensibilidade:** Novos comportamentos (como retentar caminho via RRT em caso de falha de rota) podem ser plugados facilmente como subárvores sem quebrar a lógica existente.
3. **Integração SCADA:** Os nós folha enviam seus status (`RUNNING`, `SUCCESS`, `FAILURE`) diretamente para o supervisório, permitindo visualização gráfica da árvore de decisão no dashboard do operador.